# Indian Cattle & Buffalo Breed Classifier — v2 (anti-overfitting)

**Changes vs v1** (v1: efficientnet_b3, test acc 76.8%, macro-F1 0.74, train 98.6% → heavy overfitting):

1. `image_size=300` (b3 native resolution)
2. `dropout=0.4`, `weight_decay=0.01`
3. RandAugment + RandomErasing in train transforms
4. Class imbalance handled via **WeightedRandomSampler** (sqrt-inverse frequency) — pairs correctly with MixUp; falls back to class-weighted CE if MixUp is off
5. Test-time augmentation (image + horizontal flip)
6. **MixUp/CutMix** via timm (strongest regularizer)

Also: warmup-free two-phase schedule (3 frozen epochs → cosine fine-tune), grad clipping, early stopping patience 7.

> **Before closing the session:** download `best_model.pt`, `label_map.json` and the plots from `/kaggle/working` (or use *Save Version*), or they're lost.


In [ ]:
import os, json, time, random, math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
import matplotlib
import matplotlib.pyplot as plt
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPUs:", torch.cuda.device_count())
print("timm:", timm.__version__, "| torch:", torch.__version__)

In [ ]:
import os, shutil

# find the actual dataset folder (the one containing train/val/test)
SRC = None
for dp, dns, _ in os.walk("/kaggle/input"):
    if {"train", "val", "test"}.issubset(set(dns)):
        SRC = dp
        break
assert SRC, "No train/val/test folder found under /kaggle/input — check the dataset is attached"
print("real dataset path:", SRC)

# copy to a writable location so we can clean it
DATA_DIR = "/kaggle/working/data"
if not os.path.isdir(DATA_DIR):
    shutil.copytree(SRC, DATA_DIR)
    print("copied ->", DATA_DIR)
else:
    print("already present ->", DATA_DIR)

print("USE:  DATA_DIR =", DATA_DIR)

In [ ]:
import os
from PIL import Image

DATA_DIR = "/kaggle/input/datasets/siddharthranjansingh/dataset/BtrDataset"   # the writable copy you train on
bad = 0; ok = 0
for sp in ["train", "val", "test"]:
    for c in os.listdir(os.path.join(DATA_DIR, sp)):
        d = os.path.join(DATA_DIR, sp, c)
        if not os.path.isdir(d): continue
        for f in list(os.listdir(d)):
            p = os.path.join(d, f)
            try:
                Image.open(p).convert("RGB").load()   # force full decode
                ok += 1
            except Exception as e:
                os.remove(p); bad += 1
                print("removed:", p, "|", type(e).__name__)
print(f"\nscanned OK {ok}, removed {bad} unreadable images")

In [ ]:
CFG = {
    "model_name":     "tf_efficientnet_b3.ns_jft_in1k",
    "image_size":     300,          # (1) b3 native res
    "batch_size":     32,
    "num_workers":    4,
    "dropout":        0.4,          # (2)
    "weight_decay":   0.01,         # (2)
    "label_smoothing": 0.1,
    "freeze_epochs":  3,
    "total_epochs":   45,
    "head_lr":        1e-3,         # phase 1 (frozen backbone)
    "ft_backbone_lr": 1e-4,         # phase 2
    "ft_head_lr":     5e-4,
    "min_lr":         1e-6,
    "patience":       10,            # early stopping on val acc
    "grad_clip":      1.0,
    # (6) MixUp / CutMix
    "use_mixup":      True,
    "mixup_alpha":    0.2,
    "cutmix_alpha":   1.0,
    "mixup_prob":     0.7,
    "mixup_switch_prob": 0.5,
    # (4) rare-class handling
    "sampler_power":  0.5,          # sqrt-inverse-frequency oversampling
    "work_dir":       "/kaggle/working",
}
print(json.dumps(CFG, indent=2))

In [ ]:
# ---- Datasets: lock ALL splits to the train class order ----
# (test/ has 63 breeds — missing Holstein_Friesian — so we pin class_to_idx from train)

IMG_EXTS = (".jpg", ".jpeg", ".png")

train_classes = sorted(
    d for d in os.listdir(os.path.join(data_dir, "train"))
    if os.path.isdir(os.path.join(data_dir, "train", d))
)
class_to_idx = {c: i for i, c in enumerate(train_classes)}
num_classes = len(train_classes)
print(f"{num_classes} classes (locked to train order)")

class LockedImageFolder(datasets.ImageFolder):
    """ImageFolder with labels locked to the global train class order.

    Returns only the classes actually present in the split (test/ is missing
    Holstein_Friesian), each mapped to its TRAIN index — so labels stay
    aligned and torchvision doesn't raise FileNotFoundError on absent dirs.
    """
    def find_classes(self, directory):
        present = sorted(
            d for d in os.listdir(directory)
            if os.path.isdir(os.path.join(directory, d))
        )
        unknown = set(present) - set(class_to_idx)
        if unknown:
            raise ValueError(f"Unknown classes in {directory}: {unknown}")
        return present, {c: class_to_idx[c] for c in present}

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
S = CFG["image_size"]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),        # (3)
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(S * 1.14)),
    transforms.CenterCrop(S),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = LockedImageFolder(os.path.join(data_dir, "train"), transform=train_tf)
val_ds   = LockedImageFolder(os.path.join(data_dir, "val"),   transform=eval_tf)
test_ds  = LockedImageFolder(os.path.join(data_dir, "test"),  transform=eval_tf)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

# (4) sqrt-inverse-frequency oversampling of rare breeds
train_targets = np.array(train_ds.targets)
class_counts = np.bincount(train_targets, minlength=num_classes).astype(float)
print("images/class: min %d, max %d" % (class_counts.min(), class_counts.max()))
sample_w = (1.0 / class_counts) ** CFG["sampler_power"]
sample_w = sample_w[train_targets]
sampler = WeightedRandomSampler(
    torch.as_tensor(sample_w, dtype=torch.double),
    num_samples=len(train_ds), replacement=True,
)

dl_kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
             pin_memory=True, persistent_workers=CFG["num_workers"] > 0)
train_dl = DataLoader(train_ds, sampler=sampler, drop_last=True, **dl_kw)
val_dl   = DataLoader(val_ds,  shuffle=False, **dl_kw)
test_dl  = DataLoader(test_ds, shuffle=False, **dl_kw)

with open(os.path.join(CFG["work_dir"], "label_map.json"), "w") as f:
    json.dump({"classes": train_classes, "class_to_idx": class_to_idx}, f, indent=2)
print("label_map.json saved")

In [ ]:
# ---- Model, MixUp, losses ----
model = timm.create_model(
    CFG["model_name"], pretrained=True,
    num_classes=num_classes,
    drop_rate=CFG["dropout"],          # (2) classifier dropout
)
model = model.to(device)

mixup_fn = None
if CFG["use_mixup"]:                    # (6)
    mixup_fn = Mixup(
        mixup_alpha=CFG["mixup_alpha"], cutmix_alpha=CFG["cutmix_alpha"],
        prob=CFG["mixup_prob"], switch_prob=CFG["mixup_switch_prob"],
        label_smoothing=CFG["label_smoothing"], num_classes=num_classes,
    )
    train_criterion = SoftTargetCrossEntropy()
else:
    # fallback: class-weighted CE (weights normalized to mean 1)
    w = (1.0 / torch.as_tensor(class_counts, dtype=torch.float))
    w = (w / w.mean()).to(device)
    train_criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smoothing"])

val_criterion = nn.CrossEntropyLoss()   # honest, unweighted val loss
scaler = torch.amp.GradScaler("cuda")

def set_backbone_frozen(frozen: bool):
    head = model.get_classifier()
    head_params = set(id(p) for p in head.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_params) if frozen else True

n_params = sum(p.numel() for p in model.parameters())
print(f"{CFG['model_name']}: {n_params/1e6:.1f}M params, MixUp={'on' if mixup_fn else 'off'}")

In [ ]:
!apt-get -qq install -y tesseract-ocr >/dev/null 2>&1
!pip install pytesseract -q
import os, shutil, pytesseract
from PIL import Image

DATA_DIR = "/kaggle/working/data"
BREEDS = ["Bachaur","Shweta Kapila","Gangatari","Dagri","Badri","Nari","Kherigarh"]
MAX_CHARS = 40                      # >this many detected characters = text graphic, not a photo
junk_dir = "/kaggle/working/text_junk"

checked = removed = 0
for sp in ["train","val","test"]:
    for b in BREEDS:
        d = os.path.join(DATA_DIR, sp, b)
        if not os.path.isdir(d): continue
        for f in list(os.listdir(d)):
            if not f.startswith(f"new_{b}_"): continue   # only the re-sourced ones
            p = os.path.join(d, f); checked += 1
            try:
                txt = pytesseract.image_to_string(Image.open(p).convert("RGB"))
            except Exception:
                continue
            if sum(c.isalnum() for c in txt) > MAX_CHARS:
                dst = os.path.join(junk_dir, sp, b); os.makedirs(dst, exist_ok=True)
                shutil.move(p, os.path.join(dst, f)); removed += 1
print(f"checked {checked} re-sourced images, quarantined {removed} text-heavy ones -> {junk_dir}")

In [ ]:
# ---- Contact sheet of confident mistakes (for the manual data audit) ----
import torchvision.transforms.functional as TF
from PIL import Image

AUDIT = ["Bachaur","Shweta Kapila","Gangatari","Dagri","Badri","Nari","Kherigarh"]
audit_idx = {train_classes.index(b) for b in AUDIT if b in train_classes}

model.eval()
rows = []
with torch.no_grad():
    for path, y in test_ds.samples:            # (filepath, label) pairs
        if y not in audit_idx:
            continue
        img = eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        with torch.cuda.amp.autocast():
            p = torch.softmax(model(img) + model(torch.flip(img, [3])), 1)[0]  # TTA
        conf, pred = p.max(0)
        if pred.item() != y:                    # wrong
            rows.append((float(conf), path, train_classes[y], train_classes[pred.item()]))

rows.sort(reverse=True)                          # most confident mistakes first
print(f"{len(rows)} misclassified images in audit breeds\n")
n = min(24, len(rows))
cols = 4; import math as _m; r = _m.ceil(n/cols)
fig, ax = plt.subplots(r, cols, figsize=(16, 4*r)); ax = ax.ravel()
for i in range(n):
    conf, path, true, pred = rows[i]
    ax[i].imshow(Image.open(path).convert("RGB")); ax[i].axis("off")
    ax[i].set_title(f"{true}→{pred} ({conf:.0%})", fontsize=9, color="crimson")
for j in range(n, len(ax)): ax[j].axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/audit_mistakes.png", dpi=120); plt.show()

for conf, path, true, pred in rows[:30]:
    print(f"  {conf:.0%}  {true:>16} -> {pred:<16}  {os.path.basename(path)}")

In [ ]:
# ---- Train / eval loops ----
def train_one_epoch(optimizer):
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in train_dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        y_hard = y
        if mixup_fn is not None:
            x, y = mixup_fn(x, y)      # y becomes soft targets
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out = model(x)
            loss = train_criterion(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        bs = x.size(0)
        loss_sum += loss.item() * bs
        correct  += (out.argmax(1) == y_hard).sum().item()  # vs original labels
        seen     += bs
    return loss_sum / seen, correct / seen

@torch.no_grad()
def evaluate(dl, tta=False):
    model.eval()
    loss_sum, correct, seen = 0.0, 0, 0
    all_preds, all_targets = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            out = model(x)
            if tta:                     # (5) image + horizontal flip
                out = out + model(torch.flip(x, dims=[3]))
                out = out / 2
            loss = val_criterion(out, y)
        bs = x.size(0)
        loss_sum += loss.item() * bs
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        seen    += bs
        all_preds.append(preds.cpu()); all_targets.append(y.cpu())
    return (loss_sum / seen, correct / seen,
            torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy())

In [ ]:
# ---- Training: phase 1 (frozen head-only) -> phase 2 (fine-tune, cosine) ----
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_acc, best_epoch, bad_epochs = 0.0, -1, 0
ckpt_path = os.path.join(CFG["work_dir"], "best_model.pt")

# Phase 1
set_backbone_frozen(True)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=CFG["head_lr"], weight_decay=CFG["weight_decay"],
)
scheduler = None

for epoch in range(CFG["total_epochs"]):
    if epoch == CFG["freeze_epochs"]:   # switch to phase 2
        set_backbone_frozen(False)
        head_ids = set(id(p) for p in model.get_classifier().parameters())
        backbone = [p for p in model.parameters() if id(p) not in head_ids]
        head     = [p for p in model.parameters() if id(p) in head_ids]
        optimizer = torch.optim.AdamW(
            [{"params": backbone, "lr": CFG["ft_backbone_lr"]},
             {"params": head,     "lr": CFG["ft_head_lr"]}],
            weight_decay=CFG["weight_decay"],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG["total_epochs"] - CFG["freeze_epochs"],
            eta_min=CFG["min_lr"],
        )
        print(f"--- epoch {epoch}: backbone unfrozen, cosine schedule started ---")

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(optimizer)
    va_loss, va_acc, _, _ = evaluate(val_dl)
    if scheduler is not None:
        scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)
    history["lr"].append(lr_now)
    print(f"epoch {epoch:2d} | train {tr_loss:.3f}/{tr_acc:.3%} | "
          f"val {va_loss:.3f}/{va_acc:.3%} | lr {lr_now:.2e} | {time.time()-t0:.0f}s")

    if va_acc > best_acc:
        best_acc, best_epoch, bad_epochs = va_acc, epoch, 0
        torch.save({"model": model.state_dict(), "classes": train_classes,
                    "cfg": CFG, "val_acc": va_acc, "epoch": epoch}, ckpt_path)
        print(f"   ↑ new best ({va_acc:.3%}) saved")
    else:
        bad_epochs += 1
        if epoch >= CFG["freeze_epochs"] and bad_epochs >= CFG["patience"]:
            print(f"early stop at epoch {epoch} (best {best_acc:.3%} @ {best_epoch})")
            break

with open(os.path.join(CFG["work_dir"], "history.json"), "w") as f:
    json.dump(history, f)
print(f"Best val acc {best_acc:.3%} @ epoch {best_epoch}")

In [ ]:
# ---- Curves ----
ep = range(len(history["train_loss"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, history["train_loss"], label="train (mixup)")
axes[0].plot(ep, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(ep, history["train_acc"], label="train (vs hard labels)")
axes[1].plot(ep, history["val_acc"], label="val")
axes[1].axvline(CFG["freeze_epochs"], ls="--", c="gray", label="unfreeze")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG["work_dir"], "training_curves.png"), dpi=150)
plt.show()
print("Note: with MixUp, train loss/acc aren't comparable to v1 — watch the VAL curves.")

In [ ]:
# ---- Test evaluation with TTA ----
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val {ckpt['val_acc']:.3%})")

te_loss, te_acc, preds, targets = evaluate(test_dl, tta=True)   # (5)
macro_f1 = f1_score(targets, preds, average="macro")
print(f"\nTEST (with TTA): acc {te_acc:.3%} | macro-F1 {macro_f1:.4f}")
print("v1  baseline:    acc 76.8%   | macro-F1 0.74")
print("v2.0 (30ep,b3):  acc 77.6%   | macro-F1 0.749  (undertrained, val still rising)\n")

present = sorted(set(targets) | set(preds))
report = classification_report(
    targets, preds, labels=present,
    target_names=[train_classes[i] for i in present],
    digits=3, zero_division=0,
)
print(report)
with open(os.path.join(CFG["work_dir"], "test_report.txt"), "w") as f:
    f.write(f"TEST acc {te_acc:.4f} | macro-F1 {macro_f1:.4f} (TTA)\n\n" + report)

# previously weak breeds
print("Previously weak breeds (v1 F1): Shweta_Kapila 0.33, Motu 0.55, Nagori 0.67")
f1s = f1_score(targets, preds, labels=present, average=None, zero_division=0)
name_to_f1 = {train_classes[i]: v for i, v in zip(present, f1s)}
for b in train_classes:
    if any(k in b.lower() for k in ("shweta", "motu", "nagori")):
        print(f"  {b}: F1 {name_to_f1.get(b, float('nan')):.3f}")

cm = confusion_matrix(targets, preds, labels=present)
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(16, 14))
plt.imshow(cm_norm, cmap="Blues")
plt.colorbar(shrink=.7); plt.title(f"Confusion matrix (norm) — test acc {te_acc:.1%}")
ticks = [train_classes[i] for i in present]
plt.xticks(range(len(ticks)), ticks, rotation=90, fontsize=6)
plt.yticks(range(len(ticks)), ticks, fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(CFG["work_dir"], "confusion_matrix.png"), dpi=150)
plt.show()

# most-confused pairs
off = cm_norm.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:10]
print("\nTop confused pairs (true -> predicted):")
for i, j in pairs:
    if off[i, j] > 0:
        print(f"  {ticks[i]} -> {ticks[j]}: {off[i, j]:.1%}")

## After the run

Download from `/kaggle/working` (or **Save Version → Save & Run All**): `best_model.pt`, `label_map.json`, `history.json`, `training_curves.png`, `confusion_matrix.png`, `test_report.txt`.

**Reading the results:**
- MixUp makes train loss/acc look worse than v1 — that's expected; only val/test matter.
- If val acc is still climbing when early stopping hasn't triggered by epoch 30, raise `total_epochs` to 40 (MixUp converges slower).
- If the gap is fixed but accuracy is short of target, next levers: `tf_efficientnet_b3.ns_jft_in1k` weights (NoisyStudent, usually +1–2%), or more/cleaner images for the weak breeds via `process_data.py`.
